# Phase 4 — Specificity Engineering

This notebook demonstrates the cross-reactivity screening pipeline.
After Phase 3 produces optimized nanobody candidates, Phase 4 tests
whether they discriminate the target analyte from structurally similar
molecules in urine.

**Pipeline:**
1. Screen each candidate against the cross-reactivity panel via AF3
2. Score off-target complexes with the same composite function
3. Classify as specific / borderline / cross-reactive
4. Rescue borderline candidates via negative design
5. Visualize selectivity profiles

**Note:** This notebook uses simulated data to demonstrate the analysis.
Real runs require AlphaFold 3 predictions against each off-target compound.

In [ ]:
import logging
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from omegaconf import OmegaConf

from nanolfa.filters.specificity import (
    SpecificityResult,
    OffTargetPrediction,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

OUTPUT_DIR = Path('../data/results/specificity')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load target config for panel info
pdg_cfg = OmegaConf.load('../configs/targets/pdg.yaml')
panel = list(pdg_cfg.target.cross_reactivity_panel)
compound_names = [c['name'] for c in panel]

print(f'Cross-reactivity panel for PdG: {len(panel)} compounds')
for c in panel:
    print(f'  {c["name"]:<30} ({c["role"]}) max CR: {c["max_cross_reactivity_pct"]}%')

## 1. Simulated Cross-Reactivity Data

We generate synthetic specificity results to demonstrate the analysis.
In production, these come from actual AF3 predictions against each
off-target compound.

In [ ]:
rng = np.random.default_rng(42)
n_candidates = 20

sim_results = []
for i in range(n_candidates):
    on_target = 0.65 + rng.normal(0, 0.08)
    on_target = float(np.clip(on_target, 0.45, 0.92))

    ot_predictions = []
    for compound in panel:
        # Aglycone is most similar — highest cross-reactivity risk
        if compound['role'] == 'aglycone':
            ot_score = on_target * rng.uniform(0.3, 0.7)
        elif compound['role'] == 'positional_isomer':
            ot_score = on_target * rng.uniform(0.2, 0.6)
        else:
            ot_score = on_target * rng.uniform(0.05, 0.3)

        ot_predictions.append(OffTargetPrediction(
            compound_name=compound['name'],
            compound_role=compound.get('role', 'unknown'),
            compound_smiles=compound['smiles'],
            max_allowed_cross_reactivity_pct=compound.get('max_cross_reactivity_pct', 10),
            composite_score=float(np.clip(ot_score, 0, 1)),
            iptm=float(np.clip(ot_score * 0.9, 0, 1)),
            binding_energy=float(-ot_score * 40),
        ))

    worst = max(ot_predictions, key=lambda p: p.composite_score)
    mean_ot = np.mean([p.composite_score for p in ot_predictions])
    selectivity = on_target / max(worst.composite_score, 0.01)

    failure_reasons = []
    if worst.composite_score > 0.50:
        failure_reasons.append(f'off-target {worst.compound_name}: {worst.composite_score:.3f}')
    if selectivity < 2.0:
        failure_reasons.append(f'selectivity={selectivity:.2f}')

    if not failure_reasons:
        tier = 'specific'
    elif selectivity >= 1.5:
        tier = 'borderline'
    else:
        tier = 'cross-reactive'

    sim_results.append(SpecificityResult(
        candidate_id=f'candidate_{i:03d}',
        on_target_score=on_target,
        off_target_predictions=ot_predictions,
        worst_off_target=worst.compound_name,
        worst_off_target_score=worst.composite_score,
        selectivity_ratio=float(selectivity),
        mean_off_target_score=float(mean_ot),
        passes=tier == 'specific',
        tier=tier,
        failure_reasons=failure_reasons,
        rescue_candidate=tier == 'borderline',
    ))

# Summary
for tier_name in ['specific', 'borderline', 'cross-reactive']:
    count = sum(1 for r in sim_results if r.tier == tier_name)
    print(f'{tier_name}: {count}')

## 2. Cross-Reactivity Heatmap

Visualize the binding scores of each candidate against every compound
in the panel. The on-target column should be consistently dark (high score),
while off-target columns should be light (low score).

In [ ]:
# Build the cross-reactivity matrix
candidate_ids = [r.candidate_id for r in sim_results]
all_columns = ['ON-TARGET'] + compound_names

matrix = np.zeros((len(sim_results), len(all_columns)))
for i, r in enumerate(sim_results):
    matrix[i, 0] = r.on_target_score
    for j, cn in enumerate(compound_names):
        pred = next((p for p in r.off_target_predictions if p.compound_name == cn), None)
        matrix[i, j + 1] = pred.composite_score if pred else 0

# Sort by selectivity ratio
sort_idx = np.argsort([r.selectivity_ratio for r in sim_results])[::-1]
matrix_sorted = matrix[sort_idx]
ids_sorted = [candidate_ids[i] for i in sort_idx]
tiers_sorted = [sim_results[i].tier for i in sort_idx]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(matrix_sorted, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.8)

ax.set_xticks(range(len(all_columns)))
ax.set_xticklabels(all_columns, rotation=45, ha='right', size=9)
ax.set_yticks(range(len(ids_sorted)))

tier_colors = {'specific': '#4CAF50', 'borderline': '#FF9800', 'cross-reactive': '#F44336'}
ylabels = [f'{cid} [{tier[0].upper()}]' for cid, tier in zip(ids_sorted, tiers_sorted)]
ax.set_yticklabels(ylabels, size=8)
for i, tier in enumerate(tiers_sorted):
    ax.get_yticklabels()[i].set_color(tier_colors[tier])

# Add score text
for i in range(matrix_sorted.shape[0]):
    for j in range(matrix_sorted.shape[1]):
        val = matrix_sorted[i, j]
        color = 'white' if val > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', size=7, color=color)

# Highlight the on-target column
ax.axvline(x=0.5, color='blue', linewidth=2, alpha=0.3)

plt.colorbar(im, ax=ax, label='Composite Score', shrink=0.6)
ax.set_title('Cross-Reactivity Matrix\n(sorted by selectivity ratio, descending)', size=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cross_reactivity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Selectivity Ratio Analysis

The selectivity ratio (on-target score / worst off-target score) is the
primary specificity metric. Ratio > 2.0 passes; 1.5–2.0 is borderline
and eligible for negative design rescue.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Selectivity ratio bar chart
ax = axes[0]
ratios = [r.selectivity_ratio for r in sim_results]
colors = [tier_colors[r.tier] for r in sim_results]
sort_idx = np.argsort(ratios)[::-1]

ax.barh(
    range(len(ratios)),
    [ratios[i] for i in sort_idx],
    color=[colors[i] for i in sort_idx],
    height=0.7,
)
ax.axvline(x=2.0, color='green', linestyle='--', alpha=0.7, label='Pass (>2.0)')
ax.axvline(x=1.5, color='orange', linestyle='--', alpha=0.7, label='Borderline (1.5)')
ax.set_yticks(range(len(ratios)))
ax.set_yticklabels([sim_results[i].candidate_id for i in sort_idx], size=8)
ax.set_xlabel('Selectivity Ratio', size=12)
ax.set_title('Selectivity Ratio by Candidate', size=13)
ax.legend(loc='lower right')
ax.invert_yaxis()

# On-target vs worst off-target scatter
ax = axes[1]
on_targets = [r.on_target_score for r in sim_results]
off_targets = [r.worst_off_target_score for r in sim_results]

for tier_name, color in tier_colors.items():
    mask = [r.tier == tier_name for r in sim_results]
    ax.scatter(
        [ot for ot, m in zip(on_targets, mask) if m],
        [of for of, m in zip(off_targets, mask) if m],
        c=color, label=tier_name, s=60, alpha=0.8, edgecolors='white',
    )

# Add selectivity lines
x_line = np.linspace(0.3, 1.0, 100)
ax.plot(x_line, x_line / 2.0, '--', color='green', alpha=0.5, label='2.0x selectivity')
ax.plot(x_line, x_line / 1.5, '--', color='orange', alpha=0.5, label='1.5x selectivity')
ax.plot(x_line, x_line, ':', color='red', alpha=0.3, label='1:1 (no selectivity)')

ax.set_xlabel('On-Target Composite Score', size=12)
ax.set_ylabel('Worst Off-Target Score', size=12)
ax.set_title('On-Target vs Off-Target Scores', size=13)
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(0.3, 1.0)
ax.set_ylim(0, 0.7)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'selectivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Per-Compound Cross-Reactivity Profile

Which off-target compounds are most problematic? Aglycones and positional
isomers are typically the hardest to discriminate because they share the
steroid ring system.

In [ ]:
# Box plot of off-target scores per compound
fig, ax = plt.subplots(figsize=(12, 5))

compound_data = {cn: [] for cn in compound_names}
for r in sim_results:
    for p in r.off_target_predictions:
        compound_data[p.compound_name].append(p.composite_score)

bp = ax.boxplot(
    [compound_data[cn] for cn in compound_names],
    labels=[cn.replace('_', '\n') for cn in compound_names],
    patch_artist=True,
)

# Color by role
role_colors = {
    'aglycone': '#F44336', 'structural_analog': '#FF9800',
    'positional_isomer': '#FF5722', 'androgen_metabolite': '#2196F3',
    'corticosteroid': '#4CAF50',
}
for patch, compound in zip(bp['boxes'], panel):
    color = role_colors.get(compound.get('role', ''), '#9E9E9E')
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.axhline(y=0.50, color='red', linestyle='--', alpha=0.5, label='Rejection threshold')
ax.set_ylabel('Off-Target Composite Score', size=12)
ax.set_title('Cross-Reactivity by Compound (across all candidates)', size=13)
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'per_compound_crossreactivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Negative Design Concept

For borderline candidates, the negative-design rescue attempts to weaken
off-target binding while preserving on-target binding.

**Strategy:**
1. Compare binding contacts: on-target vs off-target
2. Identify CDR residues that contact the off-target but NOT the on-target
3. Penalize those residues in ProteinMPNN redesign
4. Re-predict both complexes
5. Accept if: on-target drops < 5% AND off-target improves > 20%

In [ ]:
# Simulate rescue results for borderline candidates
borderline = [r for r in sim_results if r.tier == 'borderline']
print(f'Borderline candidates eligible for rescue: {len(borderline)}')

if borderline:
    print(f'\n{"Candidate":<20} {"On-target":>10} {"Worst OT":>10} {"Selectivity":>12} {"Worst compound"}')
    print('-' * 75)
    for r in borderline:
        print(
            f'{r.candidate_id:<20} {r.on_target_score:>10.3f} '
            f'{r.worst_off_target_score:>10.3f} {r.selectivity_ratio:>12.2f} '
            f'{r.worst_off_target}'
        )
    
    print('\nIn production, run:')
    print('  python scripts/screen_crossreactivity.py --candidates ... --config ... --rescue')

## 6. Final Selection

After Phase 4, the candidate pool is filtered to only specific binders.
These proceed to Phase 5 (LFA optimization) and ultimately to
experimental validation.

In [ ]:
specific = [r for r in sim_results if r.tier == 'specific']

print(f'Candidates passing Phase 4: {len(specific)}/{len(sim_results)}')
print(f'\nTop 10 by selectivity ratio:')
print(f'{"Candidate":<20} {"On-target":>10} {"Selectivity":>12} {"Mean OT":>10}')
print('-' * 55)

for r in sorted(specific, key=lambda x: x.selectivity_ratio, reverse=True)[:10]:
    print(
        f'{r.candidate_id:<20} {r.on_target_score:>10.3f} '
        f'{r.selectivity_ratio:>12.1f}x {r.mean_off_target_score:>10.3f}'
    )

print(f'\nThese candidates proceed to Phase 5 (LFA optimization).')

---

**Next:** Phase 5 — LFA Optimization (notebook `05_lfa_optimization.ipynb`)